In [2]:
import sys
sys.path.append("/kaggle/input/datasets/artyomstep/train-dl2-hse-hw1")

from train import *

In [3]:
# Сначала сделаем через планировщик, посмотрим что там да как работает

In [4]:
TRAIN_DATA = "https://huggingface.co/datasets/puhsu/hw01-data/resolve/main/train_dataset.pt"
VAL_DATA = "https://huggingface.co/datasets/puhsu/hw01-data/resolve/main/val_dataset.pt"

torch.manual_seed(0)

train_dataset = torch.utils.data.TensorDataset(*map(torch.nan_to_num, load_from_url(TRAIN_DATA)))
val_dataset = torch.utils.data.TensorDataset(*map(torch.nan_to_num, load_from_url(VAL_DATA)))

Y_mean = train_dataset.tensors[1].mean()
Y_std = train_dataset.tensors[1].std()
train_dataset.tensors = (train_dataset.tensors[0], (train_dataset.tensors[1] - Y_mean) / Y_std)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
device = torch.device('cuda:0')
model.to(device)

train_dl = torch.utils.data.DataLoader(train_dataset, num_workers=0, batch_size=16, shuffle=True)
val_dl = torch.utils.data.DataLoader(val_dataset, num_workers=0, batch_size=1024)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)



./train_dataset.pt: 630MB [00:04, 143MB/s]  
./val_dataset.pt: 236MB [00:02, 103MB/s]  


In [5]:
from torch.profiler import profile, record_function, ProfilerActivity

def train_step(x, y):
    optimizer.zero_grad()
    with record_function("x_to_device"):
        x_gpu = x.to(device)
    with record_function("forward"):
        pred = model(x_gpu)
    with record_function("loss"):
        target = y.unsqueeze(1).repeat(1, len(model.tcells)).to(device)
        loss = F.mse_loss(pred, target)
    with record_function("backward"):
        loss.backward()
    with record_function("optimizer"):
        optimizer.step()

In [6]:
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
[W916 16:20:06.509542854 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


In [7]:
# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )

Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::mul,180,2.59,2170.38
aten::copy_,102,1.45,334.62
aten::sum,255,4.81,290.52
aten::addmm,90,4.79,255.99
aten::add,153,2.32,158.46
aten::mm,180,4.79,151.78
aten::native_layer_norm_backward,72,1.25,9.99
aten::bmm,54,1.55,8.67
aten::native_layer_norm,72,1.94,5.94
aten::_softmax_backward_data,36,0.70,1.87


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::sum,255,4.81,290.52
aten::mm,180,4.79,151.78
aten::addmm,90,4.79,255.99
aten::repeat,42,4.53,0.00
aten::empty,579,3.34,0.00
aten::mse_loss,3,2.87,0.00
aten::mul,180,2.59,2170.38
aten::arange,84,2.56,0.00
aten::add,153,2.32,158.46
aten::transpose,522,2.20,0.00


In [8]:
import time

model.train()
batches = iter(train_dl)

for _ in range(10):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 1186.11 мс
Скорость: 13.49 samples/s


In [9]:
"""
что-то много всего мы умножаем, грешу на x_out = (mask.unsqueeze(-1) * x_emb).sum(dim=2), слишком большие размерности, которые потом суммируются
"""

'\nчто-то много всего мы умножаем, грешу на x_out = (mask.unsqueeze(-1) * x_emb).sum(dim=2), слишком большие размерности, которые потом суммируются\n'

In [10]:
# посмотрим на то с какими размерами тензоры у нас умножаются и тратят время (опять же для красивой таблички общался с гпт)

rows = []

for op in prof.key_averages(group_by_input_shape=True):
    if op.key == "aten::mul":
        rows.append({
            "Формы входов": str(op.input_shapes),
            "Вызовов": op.count,
            "GPU, мс": op.self_device_time_total / 1000,
        })

table = pd.DataFrame(rows)
table = table.sort_values("GPU, мс", ascending=False)

pd.set_option("display.max_colwidth", None)
display(table.head(10).round(2))

,Формы входов,Вызовов,"GPU, мс"
1,"[[16, 128, 984, 1], [16, 128, 984, 128]]",18,1855.89
7,"[[16, 128, 984, 128], [16, 128, 984, 128]]",18,227.20
6,"[[16, 128, 984, 128], [16, 128, 984, 1]]",18,83.21
8,"[[16, 128, 984], [16, 128, 984]]",18,1.81
9,"[[16, 984, 128], [16, 984, 1]]",18,1.17
0,"[[16, 984, 1], [984, 128]]",18,0.68
4,"[[16, 128, 128], [16, 128, 128]]",18,0.17
3,"[[16, 128, 128], [16, 128, 1]]",18,0.12
2,"[[16, 128, 1], [16, 128, 128]]",18,0.10
5,"[[16, 128], [16, 128]]",18,0.03


In [11]:
class TromptCell(nn.Module):
    def __init__(self, n_columns, n_prompts, d_model):
        super().__init__()
        # Embeddings (Figure 3.2)
        self.feature_emb_weight = nn.Parameter(torch.empty(n_columns, d_model))
        self.feature_emb_bias = nn.Parameter(torch.empty(n_columns, d_model))
        self.ln_emb = nn.LayerNorm(d_model)

        # Importance Getter (Figure 3.1)
        self.ln_col = nn.LayerNorm(d_model)
        self.ln_prompt = nn.LayerNorm(d_model)
        self.dense_imp = nn.Linear(2 * d_model, d_model)

        self.emb_column = nn.Parameter(torch.empty(n_columns, d_model))
        self.emb_prompt = nn.Parameter(torch.empty(n_prompts, d_model))

        # Modified expansion block (Figure 3.3)
        # Without non-linearities! This is important to make significant speed-ups possible.
        self.dense_expand = nn.Linear(1, n_prompts)

        self.reset_parameters()

    def reset_parameters(self):
        d_rsqrt = self.feature_emb_weight.shape[1] ** -0.5
        nn.init.uniform_(self.feature_emb_weight, -d_rsqrt, d_rsqrt)
        nn.init.uniform_(self.feature_emb_bias, -d_rsqrt, d_rsqrt)
        nn.init.normal_(self.emb_column, std=0.01)
        nn.init.normal_(self.emb_prompt, std=0.01)

    def forward(self, x: torch.Tensor, prev_cell_out: torch.Tensor) -> torch.Tensor:
        x_emb = x.unsqueeze(-1) * self.feature_emb_weight + self.feature_emb_bias.unsqueeze(0)
        x_emb = F.relu(x_emb)
        x_emb = self.ln_emb(x_emb)

        x_prompt = self.emb_prompt.unsqueeze(0).repeat(x_emb.shape[0], 1, 1)
        x_prompt = self.dense_imp(torch.cat([self.ln_prompt(x_prompt), prev_cell_out], dim=-1)) + x_prompt
        x_column = self.ln_col(self.emb_column.unsqueeze(0).repeat(x_emb.shape[0], 1, 1))
        mask = torch.softmax(x_prompt @ x_column.transpose(1,2), dim=-1)

        x_out = mask @ x_emb
        x_out = x_out * (1 + self.dense_expand.weight[:, 0])[None, :, None]
        x_out = x_out + mask.sum(dim=-1, keepdim=True) * self.dense_expand.bias[None, :, None]

        return x_out

In [12]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)

True


In [13]:
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

In [14]:
# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )

Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::bmm,108,2.59,16.45
aten::native_layer_norm_backward,72,1.14,9.18
aten::native_layer_norm,72,1.60,6.27
aten::sum,255,4.03,4.85
aten::mul,234,2.96,4.63
aten::add,171,2.18,3.05
aten::copy_,102,1.08,2.66
aten::mm,144,3.35,1.94
aten::_softmax_backward_data,36,0.51,1.88
aten::threshold_backward,36,0.50,1.67


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::sum,255,4.03,4.85
aten::mm,144,3.35,1.94
aten::mul,234,2.96,4.63
aten::empty,579,2.88,0.00
aten::addmm,72,2.86,1.14
aten::bmm,108,2.59,16.45
aten::add,171,2.18,3.05
aten::native_layer_norm,72,1.60,6.27
aten::as_strided,1656,1.52,0.00
aten::transpose,468,1.49,0.00


In [15]:
import time

model.train()
batches = iter(train_dl)

for _ in range(3):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 30
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 24.82 мс
Скорость: 644.76 samples/s


In [16]:
class TromptCell(nn.Module):
    def __init__(self, n_columns, n_prompts, d_model):
        super().__init__()
        # Embeddings (Figure 3.2)
        self.feature_emb_weight = nn.Parameter(torch.empty(n_columns, d_model))
        self.feature_emb_bias = nn.Parameter(torch.empty(n_columns, d_model))
        self.ln_emb = nn.LayerNorm(d_model)

        # Importance Getter (Figure 3.1)
        self.ln_col = nn.LayerNorm(d_model)
        self.ln_prompt = nn.LayerNorm(d_model)
        self.dense_imp = nn.Linear(2 * d_model, d_model)

        self.emb_column = nn.Parameter(torch.empty(n_columns, d_model))
        self.emb_prompt = nn.Parameter(torch.empty(n_prompts, d_model))

        # Modified expansion block (Figure 3.3)
        # Without non-linearities! This is important to make significant speed-ups possible.
        self.dense_expand = nn.Linear(1, n_prompts)

        self.reset_parameters()

    def reset_parameters(self):
        d_rsqrt = self.feature_emb_weight.shape[1] ** -0.5
        nn.init.uniform_(self.feature_emb_weight, -d_rsqrt, d_rsqrt)
        nn.init.uniform_(self.feature_emb_bias, -d_rsqrt, d_rsqrt)
        nn.init.normal_(self.emb_column, std=0.01)
        nn.init.normal_(self.emb_prompt, std=0.01)

    def forward(self, x: torch.Tensor, prev_cell_out: torch.Tensor) -> torch.Tensor:
        x_emb = x.unsqueeze(-1) * self.feature_emb_weight + self.feature_emb_bias.unsqueeze(0)
        x_emb = F.relu(x_emb)
        x_emb = self.ln_emb(x_emb)

        x_prompt = self.emb_prompt.unsqueeze(0).repeat(x_emb.shape[0], 1, 1)
        x_prompt = self.dense_imp(torch.cat([self.ln_prompt(x_prompt), prev_cell_out], dim=-1)) + x_prompt
        x_column = self.ln_col(self.emb_column).unsqueeze(0) # убрали повторы, и добавили новую ось, код сам все повторит
        mask = torch.softmax(x_prompt @ x_column.transpose(1,2), dim=-1)

        x_out = mask @ x_emb
        x_out = x_out * (1 + self.dense_expand.weight[:, 0])[None, :, None]
        x_out = x_out + mask.sum(dim=-1, keepdim=True) * self.dense_expand.bias[None, :, None]

        return x_out

In [17]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )

True


W0916 16:22:40.162000 58 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::_foreach_addcdiv_,3,0.42,0.56
aten::_foreach_mul_,6,0.51,0.51
aten::_foreach_lerp_,3,0.09,0.41
aten::_foreach_addcmul_,3,0.49,0.40
aten::_foreach_sqrt,3,0.60,0.34
aten::_foreach_div_,3,0.32,0.28
aten::_foreach_add_,6,0.57,0.27
aten::_foreach_copy_,6,0.11,0.02
aten::copy_,9,0.12,0.02
aten::mean,3,0.04,0.02


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::empty_strided,288,1.27,0.00
aten::_foreach_sqrt,3,0.60,0.34
aten::_foreach_add_,6,0.57,0.27
aten::item,558,0.56,0.00
aten::_foreach_mul_,6,0.51,0.51
aten::_foreach_addcmul_,3,0.49,0.40
aten::_foreach_addcdiv_,3,0.42,0.56
aten::_foreach_div_,3,0.32,0.28
aten::cat,9,0.27,0.00
aten::add_,279,0.25,0.00


In [18]:
import time

model.train()
batches = iter(train_dl)

for _ in range(10):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 13.69 мс
Скорость: 1169.01 samples/s


In [19]:
class TromptCell(nn.Module):
    def __init__(self, n_columns, n_prompts, d_model):
        super().__init__()
        # Embeddings (Figure 3.2)
        self.feature_emb_weight = nn.Parameter(torch.empty(n_columns, d_model))
        self.feature_emb_bias = nn.Parameter(torch.empty(n_columns, d_model))
        self.ln_emb = nn.LayerNorm(d_model)

        # Importance Getter (Figure 3.1)
        self.ln_col = nn.LayerNorm(d_model)
        self.ln_prompt = nn.LayerNorm(d_model)
        self.dense_imp = nn.Linear(2 * d_model, d_model)

        self.emb_column = nn.Parameter(torch.empty(n_columns, d_model))
        self.emb_prompt = nn.Parameter(torch.empty(n_prompts, d_model))

        # Modified expansion block (Figure 3.3)
        # Without non-linearities! This is important to make significant speed-ups possible.
        self.dense_expand = nn.Linear(1, n_prompts)

        self.reset_parameters()

    def reset_parameters(self):
        d_rsqrt = self.feature_emb_weight.shape[1] ** -0.5
        nn.init.uniform_(self.feature_emb_weight, -d_rsqrt, d_rsqrt)
        nn.init.uniform_(self.feature_emb_bias, -d_rsqrt, d_rsqrt)
        nn.init.normal_(self.emb_column, std=0.01)
        nn.init.normal_(self.emb_prompt, std=0.01)

    def forward(self, x: torch.Tensor, prev_cell_out: torch.Tensor) -> torch.Tensor:
        x_emb = x.unsqueeze(-1) * self.feature_emb_weight + self.feature_emb_bias.unsqueeze(0)
        x_emb = F.relu(x_emb)
        x_emb = self.ln_emb(x_emb)

        x_prompt = self.emb_prompt # опять удаляем повторы
        x_prompt = self.dense_imp(torch.cat([self.ln_prompt(x_prompt), prev_cell_out], dim=-1)) + x_prompt
        x_column = self.ln_col(self.emb_column) # убрали повторы, и добавили новую ось, код сам все повторит
        mask = torch.softmax(x_prompt @ x_column.T, dim=-1)

        x_out = mask @ x_emb
        x_out = x_out * (1 + self.dense_expand.weight[:, 0])[None, :, None]
        x_out = x_out + mask.sum(dim=-1, keepdim=True) * self.dense_expand.bias[None, :, None]

        return x_out

In [20]:
def trompt_forward(self, x):
    x_prompt = self.prompt
    outputs = []

    for cell in self.tcells:
        outputs.append(self.tdown(cell(x, x_prompt)))

    return torch.stack(outputs, dim=1).squeeze(-1)


Trompt.forward = trompt_forward

In [21]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::mm,252,6.20,6.74
aten::native_layer_norm_backward,72,1.11,4.14
aten::copy_,117,1.10,3.54
aten::sum,216,3.18,2.98
aten::native_layer_norm,72,1.49,2.58
aten::mul,234,2.94,2.54
aten::threshold_backward,36,0.48,1.68
aten::add,114,1.93,1.51
aten::clamp_min,36,0.43,1.19
aten::_foreach_addcdiv_,3,0.33,0.55


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::mm,252,6.20,6.74
aten::sum,216,3.18,2.98
aten::mul,234,2.94,2.54
aten::addmm,72,2.94,0.43
aten::empty,555,2.66,0.00
aten::add,114,1.93,1.51
aten::transpose,594,1.76,0.00
aten::native_layer_norm,72,1.49,2.58
aten::t,504,1.47,0.00
aten::add_,429,1.43,0.27


In [22]:
import time

model.train()
batches = iter(train_dl)

for _ in range(10):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 21.10 мс
Скорость: 758.19 samples/s


In [23]:
def downstream_forward(self, x):
    pw = torch.softmax(self.dense0(x).squeeze(-1), dim=-1)
    xnew = torch.bmm(pw.unsqueeze(1), x).squeeze(1)
    return self.dense_out(self.ln(F.relu(self.dense1(xnew))))


TromptDownstream.forward = downstream_forward

In [24]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::mm,252,5.96,6.84
aten::native_layer_norm_backward,72,1.12,4.16
aten::copy_,117,1.16,3.55
aten::sum,180,2.65,2.74
aten::native_layer_norm,72,1.50,2.60
aten::mul,180,2.38,2.21
aten::threshold_backward,36,0.47,1.68
aten::add,114,1.70,1.51
aten::clamp_min,36,0.45,1.19
aten::_foreach_addcdiv_,3,0.43,0.55


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::mm,252,5.96,6.84
aten::addmm,72,2.93,0.43
aten::empty,555,2.75,0.00
aten::sum,180,2.65,2.74
aten::mul,180,2.38,2.21
aten::transpose,630,1.85,0.00
aten::add,114,1.70,1.51
aten::native_layer_norm,72,1.50,2.60
aten::t,504,1.48,0.00
aten::add_,429,1.44,0.27


In [25]:
import time

model.train()
batches = iter(train_dl)

for _ in range(10):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 20.89 мс
Скорость: 766.03 samples/s


In [26]:
def trompt_forward(self, x): # деламе 1 tdown
    outputs = [] 

    for cell in self.tcells:
        outputs.append(cell(x, self.prompt))

    outputs = torch.stack(outputs, dim=1)
    outputs = outputs.flatten(0, 1)

    outputs = self.tdown(outputs)
    return outputs.reshape(x.shape[0], len(self.tcells))


Trompt.forward = trompt_forward

In [27]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::mm,162,4.07,7.09
aten::native_layer_norm_backward,57,0.88,4.33
aten::copy_,117,1.08,3.68
aten::native_layer_norm,57,1.23,2.77
aten::sum,135,1.95,2.74
aten::mul,165,2.16,2.28
aten::threshold_backward,21,0.29,1.65
aten::add,96,1.49,1.50
aten::clamp_min,21,0.22,1.17
aten::_foreach_addcdiv_,3,0.31,0.55


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::mm,162,4.07,7.09
aten::empty,465,2.36,0.00
aten::mul,165,2.16,2.28
aten::sum,135,1.95,2.74
aten::add,96,1.49,1.50
aten::native_layer_norm,57,1.23,2.77
aten::addmm,27,1.18,0.34
aten::empty_strided,288,1.16,0.00
aten::transpose,375,1.16,0.00
aten::copy_,117,1.08,3.68


In [28]:
import time

model.train()
batches = iter(train_dl)

for _ in range(10):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 15.39 мс
Скорость: 1039.77 samples/s


In [29]:
# смотрим кто чаще всего копируется и умножается

rows = []

for op in prof.key_averages(group_by_input_shape=True):
    if op.key in ["aten::mm", "aten::copy_"]:
        rows.append({
            "Операция": op.key,
            "Формы входов": op.input_shapes,
            "Вызовов": op.count,
            "GPU, мс": op.self_device_time_total / 1000,
        })

table = pd.DataFrame(rows)
table = table.sort_values("GPU, мс", ascending=False)

display(table.head(15).round(2))

,Операция,Формы входов,Вызовов,"GPU, мс"
3,aten::mm,"[[2048, 984], [984, 128]]",18,2.06
14,aten::mm,"[[128, 2048], [2048, 984]]",18,1.90
15,aten::mm,"[[2048, 128], [128, 984]]",18,1.75
20,aten::copy_,"[[16, 984, 128], [16, 984, 128], []]",18,1.58
2,aten::copy_,"[[16, 128, 984], [16, 128, 984], []]",18,1.54
4,aten::copy_,"[[16, 128, 128], [16, 128, 128], []]",36,0.47
17,aten::mm,"[[128, 984], [984, 128]]",18,0.38
18,aten::mm,"[[128, 128], [128, 256]]",36,0.28
1,aten::mm,"[[128, 128], [128, 984]]",18,0.24
16,aten::mm,"[[984, 128], [128, 128]]",18,0.24


In [30]:
# у нас сейчас tcell выполняется 6 раз, поэтому будем делать чтобы это выполнялось за 1 раз все

In [31]:
from torch.func import functional_call


def trompt_forward(self, x):
    cell_params = [dict(cell.named_parameters()) for cell in self.tcells] # собираем все вместе

    params = {}
    for name in cell_params[0]:
        params[name] = torch.stack([p[name] for p in cell_params])

    def run_cell(p):
        return functional_call(self.tcells[0], p, (x, self.prompt))

    outputs = torch.vmap(run_cell, out_dims=1)(params)

    outputs = outputs.flatten(0, 1)
    outputs = self.tdown(outputs)

    return outputs.reshape(x.shape[0], len(self.tcells))


Trompt.forward = trompt_forward

In [32]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::bmm,39,1.10,7.18
aten::mul,69,1.09,7.17
aten::sum,51,0.89,3.86
aten::native_layer_norm,21,0.43,3.28
aten::add,36,0.63,2.81
aten::native_layer_norm_backward,12,0.15,2.35
aten::threshold_backward,6,0.09,1.79
aten::clamp_min,6,0.08,1.24
aten::copy_,42,0.40,1.15
aten::_foreach_addcdiv_,3,0.32,0.57


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::empty_strided,306,1.25,0.00
aten::bmm,39,1.10,7.18
aten::mul,69,1.09,7.17
aten::select,354,0.90,0.00
aten::sum,51,0.89,3.86
aten::view,219,0.69,0.00
aten::item,558,0.68,0.00
aten::cat,57,0.65,0.24
aten::add,36,0.63,2.81
aten::empty,84,0.55,0.00


In [33]:
import time

model.train()
batches = iter(train_dl)

for _ in range(10):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 14.03 мс
Скорость: 1140.77 samples/s


In [34]:
# добавим компайл

In [35]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::_foreach_addcdiv_,3,0.34,0.55
aten::_foreach_mul_,6,0.52,0.53
aten::_foreach_lerp_,3,0.10,0.41
aten::_foreach_addcmul_,3,0.34,0.40
aten::_foreach_sqrt,3,0.54,0.33
aten::_foreach_div_,3,0.28,0.28
aten::_foreach_add_,6,0.52,0.27
aten::copy_,45,0.39,0.22
aten::_foreach_copy_,6,0.11,0.02
aten::mean,3,0.05,0.01


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::empty_strided,324,1.40,0.00
aten::item,558,0.72,0.00
aten::_foreach_sqrt,3,0.54,0.33
aten::_foreach_add_,6,0.52,0.27
aten::_foreach_mul_,6,0.52,0.53
aten::copy_,45,0.39,0.22
aten::_foreach_addcmul_,3,0.34,0.40
aten::_foreach_addcdiv_,3,0.34,0.55
aten::add_,279,0.28,0.00
aten::_foreach_div_,3,0.28,0.28


In [36]:
import time

model.train()
batches = iter(train_dl)

for _ in range(10):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 8.41 мс
Скорость: 1902.39 samples/s


In [37]:
class TromptCell(nn.Module):
    def __init__(self, n_columns, n_prompts, d_model):
        super().__init__()
        # Embeddings (Figure 3.2)
        self.feature_emb_weight = nn.Parameter(torch.empty(n_columns, d_model))
        self.feature_emb_bias = nn.Parameter(torch.empty(n_columns, d_model))
        self.ln_emb = nn.LayerNorm(d_model)

        # Importance Getter (Figure 3.1)
        self.ln_col = nn.LayerNorm(d_model)
        self.ln_prompt = nn.LayerNorm(d_model)
        self.dense_imp = nn.Linear(2 * d_model, d_model)

        self.emb_column = nn.Parameter(torch.empty(n_columns, d_model))
        self.emb_prompt = nn.Parameter(torch.empty(n_prompts, d_model))

        # Modified expansion block (Figure 3.3)
        # Without non-linearities! This is important to make significant speed-ups possible.
        self.dense_expand = nn.Linear(1, n_prompts)

        self.reset_parameters()

    def reset_parameters(self):
        d_rsqrt = self.feature_emb_weight.shape[1] ** -0.5
        nn.init.uniform_(self.feature_emb_weight, -d_rsqrt, d_rsqrt)
        nn.init.uniform_(self.feature_emb_bias, -d_rsqrt, d_rsqrt)
        nn.init.normal_(self.emb_column, std=0.01)
        nn.init.normal_(self.emb_prompt, std=0.01)

    def forward(self, x: torch.Tensor, prev_cell_out: torch.Tensor) -> torch.Tensor:
        x_emb = x.unsqueeze(-1) * self.feature_emb_weight + self.feature_emb_bias.unsqueeze(0)
        x_emb = F.relu(x_emb)
        x_emb = F.layer_norm(
            x_emb,
            self.ln_emb.normalized_shape,
            eps=self.ln_emb.eps,
        )
        x_prompt = self.emb_prompt # опять удаляем повторы
        x_prompt = self.dense_imp(torch.cat([self.ln_prompt(x_prompt), prev_cell_out], dim=-1)) + x_prompt
        x_column = self.ln_col(self.emb_column) # убрали повторы, и добавили новую ось, код сам все повторит
        mask = torch.softmax(x_prompt @ x_column.T, dim=-1)

        mask_sum = mask.sum(dim=-1, keepdim=True)
        x_out = mask @ x_emb
        x_out = x_out * self.ln_emb.weight + mask_sum * self.ln_emb.bias
        x_out = x_out * (1 + self.dense_expand.weight[:, 0])[None, :, None]
        x_out = x_out + mask.sum(dim=-1, keepdim=True) * self.dense_expand.bias[None, :, None]

        return x_out

In [38]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
# model = torch.compile(model, mode="reduce-overhead") # добавляем компайл


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::bmm,39,1.07,6.82
aten::mul,84,1.26,3.26
aten::native_layer_norm,18,0.39,2.98
aten::sum,60,1.19,2.85
aten::native_layer_norm_backward,12,0.18,2.17
aten::threshold_backward,6,0.13,1.80
aten::add,39,0.64,1.77
aten::clamp_min,6,0.07,1.24
aten::copy_,42,0.44,1.13
aten::_foreach_addcdiv_,3,0.31,0.56


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::empty_strided,306,1.38,0.00
aten::mul,84,1.26,3.26
aten::sum,60,1.19,2.85
aten::bmm,39,1.07,6.82
aten::select,354,1.05,0.00
aten::view,225,0.67,0.00
aten::add,39,0.64,1.77
aten::cat,57,0.64,0.24
aten::_foreach_mul_,6,0.60,0.52
aten::item,558,0.58,0.00


In [39]:
import time

model.train()
batches = iter(train_dl)

for _ in range(50):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 12.06 мс
Скорость: 1326.90 samples/s


In [40]:
rows = []

for op in prof.key_averages():
    if op.key in [
        "data_loading", "x_to_device", "forward",
        "loss", "backward", "optimizer"
    ]:
        rows.append({
            "Этап": op.key,
            "CPU, мс": op.cpu_time_total / op.count / 1000,
            "GPU, мс": op.device_time_total / op.count / 1000,
        })

display(pd.DataFrame(rows).round(2))

,Этап,"CPU, мс","GPU, мс"
0,data_loading,0.40,0.00
1,x_to_device,0.57,0.01
2,forward,3.87,3.50
3,forward,0.00,3.95
4,loss,0.77,0.01
5,loss,0.00,0.11
6,backward,7.73,0.00
7,backward,0.00,0.00
8,optimizer,3.43,0.93
9,x_to_device,0.00,0.01


In [41]:
"""в очередной раз добавил компайл"""

'в очередной раз добавил компайл'

In [42]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::_foreach_addcdiv_,3,0.34,0.56
aten::_foreach_mul_,6,0.51,0.53
aten::_foreach_lerp_,3,0.09,0.41
aten::_foreach_addcmul_,3,0.36,0.40
aten::_foreach_sqrt,3,0.51,0.34
aten::_foreach_div_,3,0.26,0.28
aten::_foreach_add_,6,0.51,0.27
aten::copy_,45,0.48,0.23
aten::_foreach_copy_,6,0.10,0.02
aten::mean,3,0.04,0.02


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::empty_strided,324,1.47,0.00
aten::item,558,0.58,0.00
aten::_foreach_mul_,6,0.51,0.53
aten::_foreach_add_,6,0.51,0.27
aten::_foreach_sqrt,3,0.51,0.34
aten::copy_,45,0.48,0.23
aten::_foreach_addcmul_,3,0.36,0.40
aten::_foreach_addcdiv_,3,0.34,0.56
aten::select,96,0.27,0.00
aten::_foreach_div_,3,0.26,0.28


In [43]:
import time

model.train()
batches = iter(train_dl)

for _ in range(50):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 8.13 мс
Скорость: 1968.18 samples/s


In [44]:
"""добавим fused в оптимизатор"""

'добавим fused в оптимизатор'

In [45]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5, fused=True)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::_fused_adamw_,3,0.17,5.00
aten::copy_,45,0.56,0.22
aten::_foreach_copy_,6,0.10,0.02
aten::mean,3,0.05,0.02
aten::_foreach_add_,3,0.28,0.01
aten::fill_,6,0.10,0.01
aten::mse_loss_backward,6,0.08,0.01
aten::mse_loss,3,0.10,0.01
aten::to,6,0.02,0.00
aten::view,9,0.01,0.00


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::copy_,45,0.56,0.22
aten::empty_strided,45,0.35,0.00
aten::_foreach_add_,3,0.28,0.01
aten::select,96,0.27,0.00
aten::detach,243,0.22,0.00
aten::_fused_adamw_,3,0.17,5.00
aten::unsqueeze,57,0.14,0.00
aten::cat,9,0.12,0.00
aten::as_strided,174,0.12,0.00
aten::_foreach_copy_,6,0.10,0.02


In [46]:
import time

model.train()
batches = iter(train_dl)

for _ in range(50):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 8.73 мс
Скорость: 1832.29 samples/s


In [47]:
"""обновим обучение, нашел баг + немного улучшим"""

'обновим обучение, нашел баг + немного улучшим'

In [48]:
def train_step(x, y):
    optimizer.zero_grad()

    with record_function("x_to_device"):
        x_gpu = x.to(device, non_blocking=True)
        y_gpu = y.to(device, non_blocking=True)
    with record_function("forward"):
        pred = model(x_gpu)
    with record_function("loss"):
        target = y_gpu.unsqueeze(1).expand_as(pred)
        loss = F.mse_loss(pred, target)
    with record_function("backward"):
        loss.backward()
    with record_function("optimizer"):
        optimizer.step()

In [49]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5, fused=True)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::_fused_adamw_,3,0.14,4.89
aten::copy_,42,0.38,0.23
aten::_foreach_copy_,6,0.10,0.02
aten::mean,3,0.04,0.02
aten::_foreach_add_,3,0.30,0.01
aten::mse_loss,3,0.13,0.01
aten::fill_,6,0.09,0.01
aten::mse_loss_backward,6,0.08,0.01
aten::unsqueeze,51,0.08,0.00
aten::view,3,0.01,0.00


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::copy_,42,0.38,0.23
aten::empty_strided,45,0.32,0.00
aten::_foreach_add_,3,0.30,0.01
aten::detach,243,0.23,0.00
aten::select,96,0.23,0.00
aten::_fused_adamw_,3,0.14,4.89
aten::mse_loss,3,0.13,0.01
aten::_foreach_copy_,6,0.10,0.02
aten::fill_,6,0.09,0.01
aten::cat,6,0.09,0.00


In [50]:
import time

model.train()
batches = iter(train_dl)

for _ in range(50):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 100
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 8.27 мс
Скорость: 1934.08 samples/s


In [51]:
"""теперь добавим амп в обучение"""

'теперь добавим амп в обучение'

In [52]:
import torch._functorch.config

torch._functorch.config.backward_pass_autocast = "off"

scaler = torch.amp.GradScaler("cuda")


def train_step(x, y):
    optimizer.zero_grad()

    with record_function("x_to_device"):
        x_gpu = x.to(device, non_blocking=True)
        y_gpu = y.to(device, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        with record_function("forward"):
            pred = model(x_gpu)
        with record_function("loss"):
            target = y_gpu.unsqueeze(1).expand_as(pred)
            loss = F.mse_loss(pred, target)
    with record_function("backward"):
        scaler.scale(loss).backward()
    with record_function("optimizer"):
        scaler.step(optimizer)
        scaler.update()

In [53]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
scaler = torch.amp.GradScaler("cuda")
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5, fused=True)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = iter(train_dl)

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::_fused_adamw_,3,0.19,4.40
aten::_amp_foreach_non_finite_check_and_unscale_,3,0.04,0.26
aten::copy_,54,0.49,0.22
aten::_foreach_sub_,3,0.40,0.02
aten::_foreach_copy_,6,0.11,0.01
aten::mean,3,0.05,0.01
aten::mul,9,0.21,0.01
aten::fill_,12,0.12,0.01
aten::_foreach_add_,3,0.30,0.01
aten::mse_loss,6,0.11,0.01


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::empty_strided,57,0.51,0.00
aten::copy_,54,0.49,0.22
aten::_foreach_sub_,3,0.40,0.02
aten::select,96,0.32,0.00
aten::_foreach_add_,3,0.30,0.01
aten::mul,9,0.21,0.01
aten::detach,243,0.21,0.00
aten::_to_copy,18,0.20,0.00
aten::_fused_adamw_,3,0.19,4.40
aten::as_strided,153,0.13,0.00


In [54]:
import time

model.train()
batches = iter(train_dl)

for _ in range(50):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 500
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 5.05 мс
Скорость: 3166.97 samples/s


In [55]:
"""перенесем данные на гпу"""

'перенесем данные на гпу'

In [56]:
x_train, y_train = train_dataset.tensors
x_train = x_train.to(device)
y_train = y_train.to(device)

batch_size = train_dl.batch_size


def gpu_batches():
    order = torch.randperm(len(x_train), device=device)

    for i in range(0, len(x_train), batch_size):
        ids = order[i:i + batch_size]
        yield x_train[ids], y_train[ids]

In [57]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
scaler = torch.amp.GradScaler("cuda")
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5, fused=True)

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = gpu_batches()

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True
Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::_fused_adamw_,3,0.16,2.90
aten::_amp_foreach_non_finite_check_and_unscale_,3,0.05,0.25
aten::copy_,48,0.53,0.21
aten::_foreach_sub_,3,0.34,0.02
aten::_foreach_copy_,6,0.10,0.01
aten::mean,3,0.04,0.01
aten::mul,9,0.19,0.01
aten::fill_,12,0.09,0.01
aten::index,6,0.17,0.01
aten::_foreach_add_,3,0.24,0.01


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::copy_,48,0.53,0.21
aten::empty_strided,51,0.42,0.00
aten::_foreach_sub_,3,0.34,0.02
aten::_foreach_add_,3,0.24,0.01
aten::detach,243,0.21,0.00
aten::mul,9,0.19,0.01
aten::empty,12,0.18,0.00
aten::index,6,0.17,0.01
aten::_fused_adamw_,3,0.16,2.90
aten::mse_loss,6,0.10,0.01


In [58]:
import time

model.train()
batches = gpu_batches()

for _ in range(50):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 500
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 5.06 мс
Скорость: 3164.30 samples/s


In [59]:
model.train()
batches = iter(train_dl)

for _ in range(10):
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]
) as prof:
    for _ in range(5):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

prof.export_chrome_trace("/kaggle/working/trace.json")

In [60]:
import train

train.TromptCell = TromptCell

# создаем модель заново, теперь с нашим TromptCell
torch.manual_seed(0)

model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
model.to(device)
model = torch.compile(model, mode="reduce-overhead") # добавляем компайл
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-5,
    foreach=True, # сделал форич
    fused=False,
)

optimizer.step = torch.compile(optimizer.step)

scaler = torch.amp.GradScaler("cuda")

# проверим, что подхватился наш класс
print(type(model.tcells[0]) is TromptCell)
model.train()
batches = gpu_batches()

for _ in range(3): # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    for _ in range(3):
        with record_function("data_loading"):
            x, y = next(batches)

        train_step(x, y)
        prof.step()

    torch.cuda.synchronize(device)

# я тут попросил гпт чтобы он мне сделал красивые таблички после профайлера, а то читать огромную таблицу неприятно

import pandas as pd
from IPython.display import display

rows = []

for op in prof.key_averages():
    if op.key.startswith("aten::"):
        rows.append({
            "Операция": op.key,
            "Вызовов": op.count,
            "CPU, мс": op.self_cpu_time_total / 1000,
            "GPU, мс": op.self_device_time_total / 1000,
        })

df = pd.DataFrame(rows)

for device_name in ["GPU", "CPU"]:
    column = device_name + ", мс"

    table = df.sort_values(column, ascending=False).head(20)

    print("Операции на", device_name)

    display(
        table.style
        .format({"CPU, мс": "{:.2f}", "GPU, мс": "{:.2f}"})
        .bar(subset=[column], color="lightblue")
        .hide(axis="index")
    )


True


W0916 16:24:15.839000 58 torch/_logging/_internal.py:1204] [2/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored


Операции на GPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::copy_,54,0.51,0.48
aten::_amp_foreach_non_finite_check_and_unscale_,3,0.05,0.28
aten::_foreach_copy_,6,0.14,0.03
aten::mean,3,0.07,0.03
aten::fill_,9,0.06,0.02
aten::index,6,0.21,0.02
aten::mul,6,0.16,0.02
aten::reciprocal,3,0.05,0.01
aten::mse_loss,6,0.14,0.01
aten::_amp_update_scale_,3,0.04,0.01


Операции на CPU


Операция,Вызовов,"CPU, мс","GPU, мс"
aten::copy_,54,0.51,0.48
aten::empty_strided,57,0.44,0.00
aten::detach,243,0.26,0.00
aten::index,6,0.21,0.02
aten::mul,6,0.16,0.02
aten::_to_copy,18,0.14,0.00
aten::_foreach_copy_,6,0.14,0.03
aten::mse_loss,6,0.14,0.01
aten::mse_loss_backward,6,0.11,0.01
aten::new_empty_strided,36,0.10,0.00


In [61]:
import time

model.train()
batches = gpu_batches()

for _ in range(50):  # прогреваем
    x, y = next(batches)
    train_step(x, y)

torch.cuda.synchronize(device)

steps = 500
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
print(f"Скорость: {samples / elapsed:.2f} samples/s")

Время одного шага: 5.28 мс
Скорость: 3028.27 samples/s


In [62]:
print(torch.cuda.device_count())

2


In [64]:
"""общался с гпт как сделать обучение на 2 карточках, спрашивал постепенно спрашивал советы и тд"""

'общался с гпт как сделать обучение на 2 карточках, спрашивал постепенно спрашивал советы и тд'

In [81]:
%%writefile /kaggle/working/train_ddp.py
import sys
sys.path.append("/kaggle/input/datasets/artyomstep/train-dl2-hse-hw1")

from train import *
import train

import os
import time

import torch
import torch.distributed as dist
import torch.nn.functional as F
import torch._functorch.config
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import TensorDataset, DataLoader, DistributedSampler


class TromptCell(nn.Module):
    def __init__(self, n_columns, n_prompts, d_model):
        super().__init__()
        # Embeddings (Figure 3.2)
        self.feature_emb_weight = nn.Parameter(torch.empty(n_columns, d_model))
        self.feature_emb_bias = nn.Parameter(torch.empty(n_columns, d_model))
        self.ln_emb = nn.LayerNorm(d_model)

        # Importance Getter (Figure 3.1)
        self.ln_col = nn.LayerNorm(d_model)
        self.ln_prompt = nn.LayerNorm(d_model)
        self.dense_imp = nn.Linear(2 * d_model, d_model)

        self.emb_column = nn.Parameter(torch.empty(n_columns, d_model))
        self.emb_prompt = nn.Parameter(torch.empty(n_prompts, d_model))

        # Modified expansion block (Figure 3.3)
        # Without non-linearities! This is important to make significant speed-ups possible.
        self.dense_expand = nn.Linear(1, n_prompts)

        self.reset_parameters()

    def reset_parameters(self):
        d_rsqrt = self.feature_emb_weight.shape[1] ** -0.5
        nn.init.uniform_(self.feature_emb_weight, -d_rsqrt, d_rsqrt)
        nn.init.uniform_(self.feature_emb_bias, -d_rsqrt, d_rsqrt)
        nn.init.normal_(self.emb_column, std=0.01)
        nn.init.normal_(self.emb_prompt, std=0.01)

    def forward(self, x: torch.Tensor, prev_cell_out: torch.Tensor) -> torch.Tensor:
        x_emb = x.unsqueeze(-1) * self.feature_emb_weight + self.feature_emb_bias.unsqueeze(0)
        x_emb = F.relu(x_emb)
        x_emb = F.layer_norm(
            x_emb,
            self.ln_emb.normalized_shape,
            eps=self.ln_emb.eps,
        )
        x_prompt = self.emb_prompt # опять удаляем повторы
        x_prompt = self.dense_imp(torch.cat([self.ln_prompt(x_prompt), prev_cell_out], dim=-1)) + x_prompt
        x_column = self.ln_col(self.emb_column) # убрали повторы, и добавили новую ось, код сам все повторит
        mask = torch.softmax(x_prompt @ x_column.T, dim=-1)

        mask_sum = mask.sum(dim=-1, keepdim=True)
        x_out = mask @ x_emb
        x_out = x_out * self.ln_emb.weight + mask_sum * self.ln_emb.bias
        x_out = x_out * (1 + self.dense_expand.weight[:, 0])[None, :, None]
        x_out = x_out + mask.sum(dim=-1, keepdim=True) * self.dense_expand.bias[None, :, None]

        return x_out


train.TromptCell = TromptCell


def downstream_forward(self, x):
    pw = torch.softmax(self.dense0(x).squeeze(-1), dim=-1)
    xnew = torch.bmm(pw.unsqueeze(1), x).squeeze(1)
    return self.dense_out(self.ln(F.relu(self.dense1(xnew))))


TromptDownstream.forward = downstream_forward


from torch.func import functional_call


def trompt_forward(self, x):
    cell_params = [dict(cell.named_parameters()) for cell in self.tcells] # собираем все вместе

    params = {}
    for name in cell_params[0]:
        params[name] = torch.stack([p[name] for p in cell_params])

    def run_cell(p):
        return functional_call(self.tcells[0], p, (x, self.prompt))

    outputs = torch.vmap(run_cell, out_dims=1)(params)

    outputs = outputs.flatten(0, 1)
    outputs = self.tdown(outputs)

    return outputs.reshape(x.shape[0], len(self.tcells))


Trompt.forward = trompt_forward


rank = int(os.environ["LOCAL_RANK"])
world_size = int(os.environ["WORLD_SIZE"])
torch.cuda.set_device(rank)
device = torch.device(f"cuda:{rank}")
dist.init_process_group("nccl")

torch.set_num_threads(1)
torch.manual_seed(0)
torch._functorch.config.backward_pass_autocast = "off"

GLOBAL_BATCH = 96

TRAIN_DATA = "https://huggingface.co/datasets/puhsu/hw01-data/resolve/main/train_dataset.pt"

train_dataset = torch.utils.data.TensorDataset(*map(torch.nan_to_num, load_from_url(TRAIN_DATA)))

Y_mean = train_dataset.tensors[1].mean()
Y_std = train_dataset.tensors[1].std()
train_dataset.tensors = (train_dataset.tensors[0], (train_dataset.tensors[1] - Y_mean) / Y_std)

dataset = train_dataset

sampler = DistributedSampler(
    dataset,
    num_replicas=world_size,
    rank=rank,
    shuffle=True,
    seed=0,
)

loader = DataLoader(
    dataset,
    batch_size=GLOBAL_BATCH // world_size,
    sampler=sampler,
    num_workers=0,
)

model = Trompt(
    n_columns=dataset.tensors[0].shape[1],
    n_prompts=128,
    d_model=128,
    n_cycles=6,
).to(device)

model.compile(mode="reduce-overhead")

if world_size > 1:
    model = DDP(model, device_ids=[rank])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-5,
    fused=True,
)
scaler = torch.amp.GradScaler("cuda")


def train_step(x, y):
    optimizer.zero_grad(set_to_none=True)
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)

    with torch.autocast("cuda", dtype=torch.float16):
        pred = model(x)
        target = y.unsqueeze(1).expand_as(pred)
        loss = F.mse_loss(pred, target)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()


model.train()
batches = iter(loader)

for _ in range(50):
    train_step(*next(batches))

torch.cuda.synchronize(device)
dist.barrier()
torch.cuda.synchronize(device)

steps = 500
samples = 0
start = time.perf_counter()

for _ in range(steps):
    x, y = next(batches)
    train_step(x, y)
    samples += x.shape[0]

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start

total_samples = torch.tensor(samples, device=device)
total_time = torch.tensor(elapsed, device=device, dtype=torch.float64)

dist.all_reduce(total_samples, op=dist.ReduceOp.SUM)
dist.all_reduce(total_time, op=dist.ReduceOp.MAX)

if rank == 0:
    elapsed = total_time.item()
    print(f"GPU: {world_size}, общий батч: {GLOBAL_BATCH}")
    print(f"Время одного шага: {elapsed / steps * 1000:.2f} мс")
    print(f"Скорость: {total_samples.item() / elapsed:.2f} samples/s")

from torch.profiler import profile, record_function, ProfilerActivity

dist.barrier()
torch.cuda.synchronize(device)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]
) as prof:
    for _ in range(5):
        with record_function("train_step"):
            train_step(*next(batches))
        prof.step()

    torch.cuda.synchronize(device)

prof.export_chrome_trace(f"/kaggle/working/trace_rank{rank}.json")

dist.destroy_process_group()

Overwriting /kaggle/working/train_ddp.py


In [82]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py

W0916 16:45:40.934000 1713 torch/distributed/run.py:852] 
W0916 16:45:40.934000 1713 torch/distributed/run.py:852] *****************************************
W0916 16:45:40.934000 1713 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0916 16:45:40.934000 1713 torch/distributed/run.py:852] *****************************************
[W916 16:45:41.261245897 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 16:45:43.451477346 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 16:45:43.473551715 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[rank0]:W0916 16:45:54.113000 1721 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
[rank1]:W0916 16:45:54.148000 

In [86]:
"""
С помощью профайлера каждый раз искал где можно соптимизировать
Итоговый отчет: постарались максимально убрать большие неэффективные операции работы с векторами (рипиты), сделали 1 проход dcell, объединили вместе вычисления tcell через vmap сделали fuse версию адама (почти не дало прибавки), сдлали amp, 2gpu, batchsize=96
обновление батчсайза дало самый большой буст в моем случае, остальные изменения дали в среднем около 20% прироста, (кроме 2 гпу)"""

'\nС помощью профайлера каждый раз искал где можно соптимизировать\nИтоговый отчет: постарались максимально убрать большие неэффективные операции работы с векторами (рипиты), сделали 1 проход dcell, объединили вместе вычисления tcell через vmap сделали fuse версию адама (почти не дало прибавки), сдлали amp, 2gpu, batchsize=96\nобновление батчсайза дало самый большой буст в моем случае, остальные изменения дали в среднем около 20% прироста, (кроме 2 гпу)'